In [23]:
!pip install pydub

In [15]:
from pyha_analyzer import extractors
from pyha_analyzer import preprocessors
from joblib import Parallel, delayed
import librosa
import numpy as np
import scipy
import matplotlib.pyplot as plt
import os
import requests
import multiprocessing as mp

In [2]:
CLIPS_DIR = "/home/super/data/music/Liked Sounds/Location A Sand Forrest/Zoom_F3"

clips = {}
for file in os.listdir(CLIPS_DIR):
    if file.endswith(".wav"):
        path = os.path.join(CLIPS_DIR, file)
        y, sr = librosa.load(path, sr=None)
        clips[file] = (y, sr)

print(f"Loaded {len(clips)} clips")

Loaded 6 clips


In [3]:
TEMPLATES_DIR = "/home/super/data/music/templates"
os.makedirs(TEMPLATES_DIR, exist_ok=True)

XC_API = "https://www.xeno-canto.org/api/2/recordings?query=cnt:Mozambique"
response = requests.get(XC_API)
data = response.json()

print(f"Found {len(data['recordings'])} recordings")

Found 365 recordings


In [5]:
for rec in data['recordings']:  # limit for now
    file_url = rec['file']
    species = rec['gen'] + "_" + rec['sp']
    filename = os.path.join(TEMPLATES_DIR, f"{species}_{rec['id']}.mp3")
    
    if not os.path.exists(filename):
        audio_data = requests.get(file_url).content
        with open(filename, 'wb') as f:
            f.write(audio_data)
        print(f"Downloaded: {species}")

Downloaded: Guttera_pucherani
Downloaded: Ortygornis_sephaena
Downloaded: Ortygornis_sephaena
Downloaded: Pternistis_hildebrandti
Downloaded: Pternistis_afer
Downloaded: Pternistis_afer
Downloaded: Caprimulgus_pectoralis
Downloaded: Caprimulgus_pectoralis
Downloaded: Caprimulgus_pectoralis
Downloaded: Caprimulgus_poliocephalus
Downloaded: Neafrapus_boehmi
Downloaded: Apus_berliozi
Downloaded: Gallirex_porphyreolophus
Downloaded: Gallirex_porphyreolophus
Downloaded: Centropus_burchellii
Downloaded: Centropus_grillii
Downloaded: Centropus_grillii
Downloaded: Ceuthmochares_australis
Downloaded: Ceuthmochares_australis
Downloaded: Chrysococcyx_caprius
Downloaded: Chrysococcyx_klaas
Downloaded: Chrysococcyx_cupreus
Downloaded: Chrysococcyx_cupreus
Downloaded: Cercococcyx_montanus
Downloaded: Cuculus_clamosus
Downloaded: Cuculus_solitarius
Downloaded: Columba_delegorguei
Downloaded: Streptopelia_semitorquata
Downloaded: Streptopelia_semitorquata
Downloaded: Streptopelia_capicola
Downloaded: 

In [4]:
def segment_audio(y, sr, segment_length=10):
    samples_per_segment = int(segment_length * sr)
    return [y[i:i+samples_per_segment] for i in range(0, len(y), samples_per_segment)]

In [5]:
def compute_mel_spectrogram(y, sr, n_mels=128, hop_length=512):
    S = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=n_mels, hop_length=hop_length)
    S_dB = librosa.power_to_db(S, ref=np.max)
    return S_dB

first_clip_name, (y, sr) = list(clips.items())[0]
clip_spec = compute_mel_spectrogram(y, sr)

print(f"Spectrogram shape for {first_clip_name}: {clip_spec.shape}")

Spectrogram shape for 030525_001_20250503_163053.wav: (128, 111770)


In [6]:
def match_template(clip_spec, template_spec):
    clip_norm = (clip_spec - np.mean(clip_spec)) / (np.std(clip_spec) + 1e-6)
    temp_norm = (template_spec - np.mean(template_spec)) / (np.std(template_spec) + 1e-6)
    
    corr = scipy.signal.correlate2d(clip_norm, temp_norm, mode='valid')
    score = np.max(corr)
    return score

In [7]:
def dtw_distance(clip_spec, template_spec):
    clip_mfcc = librosa.feature.mfcc(S=clip_spec, n_mfcc=13)
    temp_mfcc = librosa.feature.mfcc(S=template_spec, n_mfcc=13)
    
    D, wp = librosa.sequence.dtw(clip_mfcc, temp_mfcc, subseq=True)
    return D[-1, -1]

In [9]:
print(mp.cpu_count())

32


In [16]:
MATCH_THRESHOLD = 100

def analyze_clips(clips, templates, match_threshold=MATCH_THRESHOLD, segment_length=10):
    num_cores = mp.cpu_count()
    print(f"Using {num_cores} CPU cores for analysis...")
    results = {}

    for clip_name, (y, sr) in clips.items():
        print(f"\nAnalyzing {clip_name}...")

        if sr != 16000:
            y = librosa.resample(y, orig_sr=sr, target_sr=16000)
            sr = 16000

        segments = segment_audio(y, sr, segment_length)
        print(f"Split into {len(segments)} segments of {segment_length}s each")

        clip_results = []

        for idx, seg in enumerate(segments):
            clip_spec = compute_mel_spectrogram(seg, sr)

            segment_scores = Parallel(n_jobs=num_cores)(
                delayed(match_template)(clip_spec, temp_spec) for species, temp_spec in templates.items()
            )

            species_scores = list(zip(templates.keys(), segment_scores))
            species_scores.sort(key=lambda x: x[1], reverse=True)

            top_matches = species_scores[:10]
            detected_species = [s for s, sc in species_scores if sc >= match_threshold]

            clip_results.append({
                "segment": idx,
                "top_matches": top_matches,
                "detected_species": detected_species
            })

        results[clip_name] = clip_results
        print(f"Finished {clip_name} ({len(segments)} segments)")

    return results

In [11]:
from pydub import AudioSegment

def load_mp3_as_array(path):
    audio = AudioSegment.from_file(path, format="mp3")
    y = np.array(audio.get_array_of_samples(), dtype=np.float32)
    y = y / (2**15)
    sr = audio.frame_rate
    return y, sr

In [13]:
templates = {}

for file in os.listdir(TEMPLATES_DIR):
    if file.endswith(".mp3"):
        path = os.path.join(TEMPLATES_DIR, file)
        try:
            y, sr = librosa.load(path, sr=None)
            temp_spec = compute_mel_spectrogram(y, sr)
            species = file.split("_")[0]
            templates[species] = temp_spec
        except Exception as e:
            print(f"Skipping {file}: {e}")

Note: Illegal Audio-MPEG-Header 0x00000000 at offset 3024.
/var/tmp/ipykernel_71022/4008378409.py:7: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(path, sr=None)
/home/super/.local/lib/python3.9/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)
Note: Trying to resync...
Note: Hit end of (available) data during resync.


Skipping Acrocephalus_griseldis_183951.mp3: 


Note: Illegal Audio-MPEG-Header 0x00000000 at offset 3024.
Note: Trying to resync...
Note: Hit end of (available) data during resync.
/var/tmp/ipykernel_71022/4008378409.py:7: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(path, sr=None)
/home/super/.local/lib/python3.9/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Skipping Macronyx_croceus_452897.mp3: 


/var/tmp/ipykernel_71022/4008378409.py:7: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(path, sr=None)
Note: Illegal Audio-MPEG-Header 0x00000000 at offset 3024.
Note: Trying to resync...
Note: Hit end of (available) data during resync.
/home/super/.local/lib/python3.9/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


...

Skipping Nicator_gularis_520532.mp3: 


Note: Illegal Audio-MPEG-Header 0x50455441 at offset 304191.
Note: Trying to resync...
Note: Hit end of (available) data during resync.
Note: Illegal Audio-MPEG-Header 0x00000000 at offset 3024.
Note: Trying to resync...
Note: Hit end of (available) data during resync.
/var/tmp/ipykernel_71022/4008378409.py:7: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(path, sr=None)
/home/super/.local/lib/python3.9/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Skipping Artisornis_moreaui_345933.mp3: 


Note: Illegal Audio-MPEG-Header 0x00000000 at offset 3024.
Note: Trying to resync...
Note: Hit end of (available) data during resync.
/var/tmp/ipykernel_71022/4008378409.py:7: UserWarning: PySoundFile failed. Trying audioread instead.
  y, sr = librosa.load(path, sr=None)
/home/super/.local/lib/python3.9/site-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Skipping Andropadus_importunus_104020.mp3: 


In [17]:
results = analyze_clips(clips, templates)

Using 32 CPU cores for analysis...

Analyzing 030525_001_20250503_163053.wav...
Split into 60 segments of 10s each
Finished 030525_001_20250503_163053.wav (60 segments)

Analyzing 030525_004_20250503_183415.wav...
Split into 59 segments of 10s each
Finished 030525_004_20250503_183415.wav (59 segments)

Analyzing 030525_003_20250503_180900.wav...
Split into 52 segments of 10s each
Finished 030525_003_20250503_180900.wav (52 segments)

Analyzing 030525_005_20250503_190900.wav...
Split into 55 segments of 10s each
Finished 030525_005_20250503_190900.wav (55 segments)

Analyzing 030525_005_20250503_192700.wav...
Split into 64 segments of 10s each
Finished 030525_005_20250503_192700.wav (64 segments)

Analyzing 030525_003_20250503_174500.wav...
Split into 26 segments of 10s each
Finished 030525_003_20250503_174500.wav (26 segments)
